### S10 - RUTINA INICIAL - Genera FORECAST a partir de estado 10 - PLANIFICADO

De acuerdo a los parámetros especificados, recorre los que tengan estado 10 para proceder a realizar el FORECAST.

Luego Acutaliza a estado 20

### RUTINA INICIAL - FORECAST

1) Leer archivo Solicitudes_Compra
2) Leer datos adicionales y id relacionados
3) Leer datos adicionales de la T710_Estadis_Reposición
4) Generar GRAFICOS
5) Actulizar Estado en connexa

## A) Recopilación de Funciones

Se compilan los Algoritmos Probados

In [1]:
# LIBRERIAS NECESARIAS 
import base64
from io import BytesIO
import pandas as pd
import matplotlib.pyplot as plt
# LIBRERIAS NECESARIAS 
from datetime import datetime, timedelta
import numpy as np
from dotenv import dotenv_values
import psycopg2 as pg2    # Conectores para Postgres
import pyodbc  # Conector para SQL Server
import time  # Para medir el tiempo de ejecución
import getpass  # Para obtener el usuario del sistema operativo
import uuid  # Importar la librería uuid
# Mostrar el DataFrame resultante
import ace_tools_open as tools

# Evitar Mensajes Molestos
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category= FutureWarning)

secrets = dotenv_values(".env")   # Connection String from .env
folder = secrets["FOLDER_DATOS"]

from datetime import datetime, timedelta

# Liberias para Algoritmos
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.holtwinters import Holt


In [2]:
# Solo importa lo necesario desde el módulo de funciones
from funciones_forecast import (
    get_forecast,

    Procesar_ALGO_01,
    Procesar_ALGO_02,
    Procesar_ALGO_03,
    Procesar_ALGO_04,
    Procesar_ALGO_05,
    Procesar_ALGO_06,    

    get_execution_execute_by_status,
    get_full_parameters,
    update_execution,
    update_execution_execute,
    Open_Connection,
    Open_Diarco_Data,
    Close_Connection
)

# FUNCIONES LOCALES

### RUTINAS EXPORTADAS A FUNCIONES_FORECAST

In [ ]:

def generar_datos_PG(id_proveedor, etiqueta, ventana):
    from dotenv import dotenv_values
    import pandas as pd
    import os
    from funciones_forecast import Open_Diarco_Data, Close_Connection

    secrets = dotenv_values(".env")
    folder = secrets["FOLDER_DATOS"]

    try:
        data = pd.read_csv(f'{folder}/{etiqueta}.csv')
        data['Codigo_Articulo'] = data['Codigo_Articulo'].astype(int)
        data['Sucursal'] = data['Sucursal'].astype(int)
        data['Fecha'] = pd.to_datetime(data['Fecha'])

        articulos = pd.read_csv(f'{folder}/{etiqueta}_articulos.csv')
        print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {etiqueta}")
        return data, articulos

    except FileNotFoundError:
        print(f"-> Generando datos para ID: {id_proveedor}, Label: {etiqueta}")
        conn = Open_Diarco_Data()

        # --- ARTÍCULOS ---
        query_articulos = f"""
            SELECT *
            FROM src.base_forecast_articulos
            WHERE c_proveedor_primario = {id_proveedor}
            ORDER BY c_articulo, c_sucu_empr;
        """
        articulos = pd.read_sql(query_articulos, conn)
        if articulos.empty:
            print(f"❗ No se encontraron artículos para el proveedor {id_proveedor}.")
            Close_Connection(conn)
            return None, None

        articulos.columns = articulos.columns.str.upper()
        articulos['C_PROVEEDOR_PRIMARIO'] = articulos['C_PROVEEDOR_PRIMARIO'].astype(int)
        articulos['C_ARTICULO'] = articulos['C_ARTICULO'].astype(int)
        articulos['C_FAMILIA'] = articulos['C_FAMILIA'].astype(int)
        articulos['C_RUBRO'] = articulos['C_RUBRO'].astype(int)
        articulos['Q_DIAS_STOCK'] = articulos['Q_DIAS_STOCK'].fillna(ventana).astype(int)
        articulos['Q_DIAS_SOBRE_STOCK'] = articulos['Q_DIAS_SOBRE_STOCK'].fillna(0).astype(int)
        articulos.to_csv(f'{folder}/{etiqueta}_articulos.csv', index=False, encoding='utf-8')
        print(f"---> Datos de Artículos guardados")

        # --- VENTAS ---
        query_ventas = f"""
            SELECT fecha, codigo_articulo, sucursal, precio, costo, unidades, familia, rubro, subrubro,
                   c_proveedor_primario, nombre_articulo, clasificacion, fecha_procesado, marca_procesado
            FROM src.base_forecast_ventas
            WHERE c_proveedor_primario = {id_proveedor}
            ORDER BY fecha;
        """
        demanda = pd.read_sql(query_ventas, conn)
        if demanda.empty:
            print(f"⚠️ No se encontraron ventas para el proveedor {id_proveedor}.")
            Close_Connection(conn)
            return None, articulos

        demanda = demanda.rename(columns={
            "codigo_articulo": "Codigo_Articulo",
            "sucursal": "Sucursal",
            "fecha": "Fecha",
            "precio": "Precio",
            "costo": "Costo",
            "unidades": "Unidades",
            "familia": "Familia",
            "rubro": "Rubro",
            "subrubro": "SubRubro"
        })

        # --- MERGE ---
        data = pd.merge(
            articulos,
            demanda,
            left_on=['C_ARTICULO', 'C_SUCU_EMPR'],
            right_on=['Codigo_Articulo', 'Sucursal'],
            how='inner'
        )

        if data.empty:
            print(f"⚠️ No hay coincidencias entre artículos y ventas para el proveedor {id_proveedor}.")
            Close_Connection(conn)
            return None, articulos

        # Guardado
        data['C_ARTICULO'] = data['C_ARTICULO'].astype(int)
        data['C_SUCU_EMPR'] = data['C_SUCU_EMPR'].astype(int)
        data['Codigo_Articulo'] = data['Codigo_Articulo'].astype(int)
        data['Sucursal'] = data['Sucursal'].astype(int)
        data.to_csv(f'{folder}/{etiqueta}.csv', index=False, encoding='utf-8')
        print(f"---> Datos de RECUPERACIÓN guardados")

        # Compactar solo VENTAS
        ventas = data[['Fecha', 'Codigo_Articulo', 'Sucursal', 'Unidades']]
        ventas.to_csv(f'{folder}/{etiqueta}_Ventas.csv', index=False, encoding='utf-8')
        print(f"---> Datos de Ventas guardados")

        Close_Connection(conn)
        return data, articulos


def dividir_dataframe(data, fecha_corte):
    """
    Divide un DataFrame en dos partes: data_train y data_test según la fecha_corte.
    
    :param data: DataFrame con la columna 'Fecha'
    :param fecha_corte: Fecha límite para dividir el DataFrame (tipo datetime o string con formato 'YYYY-MM-DD')
    :return: data_train, data_test
    """
    # Asegurarse de que la columna 'Fecha' sea de tipo datetime
    data['Fecha'] = pd.to_datetime(data['Fecha'])
    
    # Filtrar los datos
    data_train = data[data['Fecha'] < pd.to_datetime(fecha_corte)]
    data_test = data[data['Fecha'] >= pd.to_datetime(fecha_corte)]
    
    return data_train, data_test

def obtener_datos_stock_PG(id_proveedor, etiqueta):
    secrets = dotenv_values(".env")   # Connection String from .env
    folder = secrets["FOLDER_DATOS"]
    
    #  Intento recuperar datos cacheados
    try:         
        print(f"-> Generando datos para ID: {id_proveedor}, Label: {etiqueta}")
        # Configuración de conexión
        # conn = Open_Connection()   # CAMBIAMOS DE ENTORNO
        conn = Open_Diarco_Data()
        
        # ----------------------------------------------------------------
        # FILTRA solo PRODUCTOS HABILITADOS y Traer datos de STOCK y PENDIENTES desde PRODUCCIÓN
        # ----------------------------------------------------------------
        query = f"""              
            SELECT codigo_proveedor, codigo_articulo, codigo_sucursal, precio_venta, precio_costo, factor_venta, stock_unidades, venta_unidades_30_dias, 
            stock_valorizado, venta_valorizada, dias_stock, f_ultima_vta, venta_unidades_1q, venta_unidades_2q
            FROM src.base_forecast_stock
            WHERE codigo_proveedor = {id_proveedor}
            ORDER BY codigo_articulo, codigo_sucursal;
        """
        # Ejecutar la consulta SQL
        df_stock = pd.read_sql(query, conn)
        file_path = f'{folder}/{etiqueta}_Stock.csv'
        df_stock['Codigo_Proveedor']= df_stock['Codigo_Proveedor'].astype(int)
        df_stock['Codigo_Articulo']= df_stock['Codigo_Articulo'].astype(int)
        df_stock['Codigo_Sucursal']= df_stock['Codigo_Sucursal'].astype(int)
        df_stock.fillna(0, inplace= True)
        # df_stock.to_csv(file_path, index=False, encoding='utf-8')        
        print(f"---> Datos de STOCK guardados: {file_path}")
        return df_stock
    except Exception as e:
        print(f"Error en get_execution: {e}")
        return None
    finally:
        Close_Connection(conn)

def obtener_demora_oc_PG(id_proveedor, etiqueta):
    secrets = dotenv_values(".env")   # Connection String from .env
    folder = secrets["FOLDER_DATOS"]
    
    #  Intento recuperar datos cacheados
    try:         
        print(f"-> Generando datos para ID: {id_proveedor}, Label: {etiqueta}")
        # Configuración de conexión
        # conn = Open_Connection()   # CAMBIAMOS DE ENTORNO
        conn = Open_Diarco_Data()
        
        # ----------------------------------------------------------------
        # FILTRA solo PRODUCTOS HABILITADOS y Traer datos de STOCK y PENDIENTES desde PRODUCCIÓN
        # ----------------------------------------------------------------
        query = f""" 
        SELECT c_oc, u_prefijo_oc, u_sufijo_oc, u_dias_limite_entrega, fecha_limite, demora, codigo_proveedor, codigo_sucursal, c_sucu_destino, c_sucu_destino_alt, 
        c_situac, f_situac, f_alta_sist, f_emision, f_entrega, c_usuario_operador
        FROM src.base_forecast_oc_demoradas
        WHERE c_proveedor = {id_proveedor};
        """
        # Ejecutar la consulta SQL
        df_demoras = pd.read_sql(query, conn)
        df_demoras['Codigo_Proveedor']= df_demoras['Codigo_Proveedor'].astype(int)
        df_demoras['Codigo_Sucursal']= df_demoras['Codigo_Sucursal'].astype(int)
        df_demoras['Demora']= df_demoras['Demora'].astype(int)
        df_demoras.fillna(0, inplace= True)         
        print(f"---> Datos de OC DEMORADAS Recuperados: {etiqueta}")
        return df_demoras
    except Exception as e:
        print(f"Error en get_execution: {e}")
        return None
    finally:
        Close_Connection(conn)

def Exportar_Pronostico(df_forecast, proveedor, etiqueta, algoritmo):
    df_forecast['Codigo_Articulo']= df_forecast['Codigo_Articulo'].astype(int)
    df_forecast['Sucursal']= df_forecast['Sucursal'].astype(int)
    
    # tools.display_dataframe_to_user(name="SET de Datos del Proveedor", dataframe=df_forecast)
    # df_forecast.info()
    #print(f'-> ** Pronostico Guardado en: {folder}/{etiqueta}_{algoritmo}_Pronostico.csv **')
    #df_forecast.to_csv(f'{folder}/{etiqueta}_{algoritmo}_Pronostico.csv', index=False)
    
    ## GUARDAR TABLA EN POSTGRES
    usuario = getpass.getuser()  # Obtiene el usuario del sistema operativo
    fecha_actual = datetime.today().strftime('%Y-%m-%d')  # Obtiene la fecha de hoy en formato 'YYYY-MM-DD'
    conn = Open_Diarco_Data()
    
    # Query de inserción
    insert_query = """
    INSERT INTO public.f_oc_precarga_connexa (
        c_proveedor, c_articulo, c_sucu_empr, q_forecast_unidades, f_alta_forecast, c_usuario_forecast, create_date
    ) VALUES (%s, %s, %s, %s, %s, %s, %s);
    """

    # Convertir el DataFrame a una lista de tuplas para la inserción en bloque
    data_to_insert = [
        (proveedor, row['Codigo_Articulo'], row['Sucursal'], row['Forecast'], fecha_actual, usuario, fecha_actual)
        for _, row in df_forecast.iterrows()
    ]

    try:
        with conn.cursor() as cur:
            cur.executemany(insert_query, data_to_insert)
        conn.commit()
        print(f"✅ Inserción completada: {len(data_to_insert)} registros insertados.")
    except Exception as e:
        conn.rollback()
        print(f"❌ Error en la inserción: {e}")
    finally:
        Close_Connection(conn)



# RUTINA PRINCIPAL para obtener el pronóstico
def get_forecast( id_proveedor, lbl_proveedor, period_lengh=30, algorithm='basic', f1=None, f2=None, f3=None, current_date=None ):
    """
    Genera la predicción de demanda según el algoritmo seleccionado.

    Parámetros:
    - id_proveedor: ID del proveedor.
    - lbl_proveedor: Etiqueta del proveedor.
    - period_lengh: Número de días del período a analizar (por defecto 30).
    - algorithm: Algoritmo a utilizar.
    - current_date: Fecha de referencia; si es None, se toma la fecha máxima de los datos.
    - factores de ponderación: F1, F2, F3  (No importa en que unidades estén, luego los hace relativos al total del peso)

    Retorna:
    - Un DataFrame con las predicciones.
    """
    
    print('Dentro del get_forecast')
    print(f'FORECAST control: {id_proveedor} - {lbl_proveedor} - ventana: {period_lengh} - {algorithm} factores: {f1} - {f2} - {f3}')
    # Generar los datos de entrada
    data, articulos = generar_datos_PG(id_proveedor, lbl_proveedor, period_lengh)

        # Determinar la fecha base
    if current_date is None:
        current_date = data['Fecha'].max()  # Se toma la última fecha en los datos
    else:
        current_date = pd.to_datetime(current_date)  # Se asegura que sea un objeto datetime
    print(f'Fecha actual {current_date}')
    

    # Selección del algoritmo de predicción
    match algorithm:
        case 'ALGO_01':
            return Procesar_ALGO_01(data, id_proveedor, lbl_proveedor, period_lengh, current_date, f1, f2, f3)  # Promedio Ponderado x 3 Factores
        case 'ALGO_02':
            return Procesar_ALGO_02(data, id_proveedor, lbl_proveedor, period_lengh, current_date) # Doble Exponencial - Modelo Holt (Tendencia)
        case 'ALGO_03':
            return Procesar_ALGO_03(data, id_proveedor, lbl_proveedor, period_lengh, current_date, f1, f2, f3) # Triple Exponencial Holt-WInter (Tendencia + Estacionalidad) (periodos, add, add)
        case 'ALGO_04':
            return Procesar_ALGO_04(data, id_proveedor, lbl_proveedor, period_lengh, current_date, f1) # EWMA con Factor alpha
        case 'ALGO_05':
            return Procesar_ALGO_05(data, id_proveedor, lbl_proveedor, period_lengh, current_date) # Promedio Venta Simple en Ventana
        case 'ALGO_06':
            return Procesar_ALGO_06(data, id_proveedor, lbl_proveedor, period_lengh, current_date) # Tendencias Ventas Semanales
        case _:
            raise ValueError(f"Error: El algoritmo '{algorithm}' no está implementado.")



# Final del MODULO FUNCIONES

In [ ]:
import pandas as pd
from datetime import date, datetime, timedelta
from funciones_forecast import Open_Diarco_Data, Close_Connection

def convertir_stock_diario_a_dict(df_stock):
    """Convierte df_stock en un diccionario {fecha: cantidad}, solo hasta ayer y con fechas válidas."""
    def es_fecha_valida(anio, mes, dia):
        try:
            return date(anio, mes, dia)
        except ValueError:
            return None

    resultado = {}
    for _, row in df_stock.iterrows():
        anio = int(row['c_anio'])
        mes = int(row['c_mes'])

        for col in df_stock.columns:
            if col.startswith("q_dia"):
                dia = int(col.replace("q_dia", ""))
                fecha_valida = es_fecha_valida(anio, mes, dia)
                if fecha_valida and fecha_valida <= (datetime.now().date() - timedelta(days=1)):
                    resultado[fecha_valida.isoformat()] = row[col]
    return resultado

def obtener_stock_como_dict(c_articulo, c_sucursal):
    """Consulta el stock y devuelve un dict {fecha: cantidad}, limitado a fechas válidas hasta ayer."""
    conn = Open_Diarco_Data()
    query_stock = f"""
        SELECT * FROM src.t710_estadis_stock
        WHERE c_anio * 100 + c_mes >= TO_CHAR(CURRENT_DATE - INTERVAL '1 month', 'YYYYMM')::INTEGER
            AND c_articulo = {c_articulo}
            AND c_sucu_empr = {c_sucursal};
    """
    df_stock = pd.read_sql(query_stock, conn)
    Close_Connection(conn)

    if df_stock.empty:
        print(f"⚠️ No se encontraron datos de stock para el artículo {c_articulo} en la sucursal {c_sucursal}.")
        return {}

    return convertir_stock_diario_a_dict(df_stock)


In [ ]:
stock_dict = obtener_stock_como_dict(c_articulo=5363, c_sucursal=1)
print(stock_dict)


In [ ]:
import pandas as pd
from datetime import date, datetime, timedelta
from funciones_forecast import Open_Diarco_Data, Close_Connection

def convertir_stock_diario(df_stock):
    """Convierte un df con columnas q_diaX en una lista de dicts con 'fecha' y 'cantidad'."""
    def es_fecha_valida(anio, mes, dia):
        try:
            return date(anio, mes, dia)
        except ValueError:
            return None

    resultado = []
    for _, row in df_stock.iterrows():
        anio = int(row['c_anio'])
        mes = int(row['c_mes'])
        for col in df_stock.columns:
            if col.startswith("q_dia"):
                dia = int(col.replace("q_dia", ""))
                fecha_valida = es_fecha_valida(anio, mes, dia)
                if fecha_valida:
                    cantidad = row[col]
                    resultado.append({
                        "fecha": fecha_valida,
                        "cantidad": cantidad
                    })

    # Filtrar fechas mayores a ayer
    ayer = datetime.now().date() - timedelta(days=1)
    resultado_filtrado = [
        {"fecha": r["fecha"].isoformat(), "cantidad": r["cantidad"]}
        for r in resultado if r["fecha"] <= ayer
    ]
    return resultado_filtrado

def obtener_stock_formateado(c_articulo, c_sucursal):
    """Consulta el stock y devuelve una lista de diccionarios fecha-cantidad válidos hasta ayer."""
    conn = Open_Diarco_Data()
    query_stock = f"""
        SELECT * FROM src.t710_estadis_stock
        WHERE c_anio * 100 + c_mes >= TO_CHAR(CURRENT_DATE - INTERVAL '1 month', 'YYYYMM')::INTEGER
            AND c_articulo = {c_articulo}
            AND c_sucu_empr = {c_sucursal};
    """
    df_stock = pd.read_sql(query_stock, conn)
    Close_Connection(conn)

    if df_stock.empty:
        print(f"⚠️ No se encontraron datos de stock para el artículo {c_articulo} en la sucursal {c_sucursal}.")
        return None

    return convertir_stock_diario(df_stock)

In [ ]:
datos = obtener_stock_formateado(c_articulo=5363, c_sucursal=1)
if datos:
    for d in datos[:5]:
        print(d)

In [ ]:
# SANDOX GRAFICOS
def convertir_stock_a_diccionario(df_stock):
    """
    Convierte un DataFrame con columnas q_diaX en una lista de diccionarios
    con claves 'fecha' y 'cantidad', donde la fecha se arma con c_anio, c_mes y el número de día.
    """
    resultado = []

    for _, row in df_stock.iterrows():
        anio = int(row['c_anio'])
        mes = int(row['c_mes'])

        for col in df_stock.columns:
            if col.startswith("q_dia"):
                dia = int(col.replace("q_dia", ""))
                fecha = f"{anio}-{mes:02d}-{dia:02d}"
                cantidad = row[col]
                resultado.append({
                    "fecha": fecha,
                    "cantidad": cantidad
                })

    return resultado



## LLAMADA LOCAL CON PARÁMETROS

In [ ]:
#----------------------------------------------------------------
# RUTINA PRINCIPAL
#----------------------------------------------------------------       

if __name__ == "__main__":
    # Aquí se inicia la ejecución programada del pronóstico
    print("🕒 Iniciando ejecución programada del FORECAST ...")
    try:
        # Ejecuta la rutina completa
        fes = get_execution_execute_by_status(10)
        for index, row in fes[fes["fee_status_id"] == 10].iterrows():
            algoritmo = row["name"]
            name = algoritmo.split('_ALGO')[0]
            method = row["method"]
            execution_id = row["forecast_execution_id"]
            id_proveedor = row["ext_supplier_code"]
            forecast_execution_execute_id = row["forecast_execution_execute_id"]
            supplier_id = row["supplier_id"]
            supply_forecast_model_id = row["forecast_model_id"]

            print(f"Procesando ejecución: {name} - Método: {method}")
            
            start_time = time.time()

            try:
                df_params = get_full_parameters(supply_forecast_model_id, execution_id) 
                ventana = 30
                f1 = f2 = f3 = None

                try:
                    if df_params is not None and not df_params.empty:
                        if len(df_params) >= 1:
                            ventana = int(float(df_params.iloc[0]['value']))
                        if len(df_params) >= 2:
                            f1 = df_params.iloc[1]['value']
                        if len(df_params) >= 3:
                            f2 = df_params.iloc[2]['value']
                        if len(df_params) >= 4:
                            f3 = df_params.iloc[3]['value']
                except Exception as e:
                    print(f"⚠️ Error interpretando parámetros: {e}")
                    ventana = 30
                    f1 = f2 = f3 = None
                
                # update_execution_execute(forecast_execution_execute_id, supply_forecast_execution_status_id=15)
                ## RUTINA PRINCIPAL
                get_forecast(id_proveedor, name, ventana, method, f1, f2, f3)
                
                # update_execution_execute(forecast_execution_execute_id, supply_forecast_execution_status_id=20)
                
                elapsed = round(time.time() - start_time, 2)
                print(f"✅ FORECAST : {algoritmo} procesado - Tiempo parcial: {elapsed} segundos")

                print("✅ Ejecución completada con éxito.")
            except Exception as e:
                print(f"❌ Error durante la ejecución del forecast: {e}")
    except Exception as e:
        print(f"❌ Error general al iniciar ejecuciones programadas: {e}")



# NUEVO SECTOR GRAFICOS CON VENTAS DIARIAS DUPLICADAS

In [4]:
 # Paths
path_ventas = f'{folder}/{name}_Ventas.csv'
path_forecast = f'{folder}/{algoritmo}_Pronostico_Extendido.csv'
path_backup = f'{folder}/{algoritmo}_Pronostico_Extendido_Con_Graficos.csv'
path_log = f'{folder}/log_graficos_{name}.txt'

# Cargar historial de ventas
df_ventas = pd.read_csv(path_ventas)
df_ventas['Codigo_Articulo'] = df_ventas['Codigo_Articulo'].astype(int)
df_ventas['Sucursal'] = df_ventas['Sucursal'].astype(int)
df_ventas['Fecha'] = pd.to_datetime(df_ventas['Fecha'])

In [ ]:
df_ventas = (
    df_ventas
    .groupby(['Fecha', 'Codigo_Articulo', 'Sucursal'], as_index=False)
    .agg({'Unidades': 'sum'})
)

In [7]:
# REVISAR SI HAY DUPLICADOS
# Buscar filas duplicadas por clave compuesta
duplicados = df_ventas[df_ventas.duplicated(subset=["Fecha", "Codigo_Articulo", "Sucursal"], keep=False)]

# Ordenar para facilitar lectura
duplicados = duplicados.sort_values(["Codigo_Articulo", "Sucursal", "Fecha"])

# Mostrar o exportar
print("⚠️ Filas duplicadas encontradas:")
print(duplicados)


⚠️ Filas duplicadas encontradas:
Empty DataFrame
Columns: [Fecha, Codigo_Articulo, Sucursal, Unidades]
Index: []


In [6]:
# 🔄 Agrupar por Fecha, Código de Artículo y Sucursal, para consolidar múltiples precios
df_ventas = (
    df_ventas
    .groupby(['Fecha', 'Codigo_Articulo', 'Sucursal'], as_index=False)
    .agg({'Unidades': 'sum'})
)

In [9]:
# # Cargar forecast extendido
# df_forecast = pd.read_csv(path_forecast)
# df_forecast.fillna(0, inplace=True)
# print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {name}")
id_proveedor = 20
# Cargar STOCK por Proveedor
"""Consulta el stock y devuelve un dict {fecha: cantidad}, limitado a fechas válidas hasta ayer."""
conn = Open_Diarco_Data()
query_stock = f"""
    SELECT DISTINCT s.*
    FROM src.t710_estadis_stock s
    LEFT JOIN src.t050_articulos a
    ON s.c_articulo = a.c_articulo
    WHERE s.c_anio * 100 + s.c_mes >= TO_CHAR(CURRENT_DATE - INTERVAL '1 month', 'YYYYMM')::INTEGER
    AND a.c_proveedor_primario = {id_proveedor};
"""
df_stock = pd.read_sql(query_stock, conn)
Close_Connection(conn)
if df_stock.empty:
    print(f"⚠️ No se encontraron datos de stock para el proveedor {id_proveedor} en el mes actual.")
   # return {}

In [10]:
articulo = 5363
sucursal = 1

In [11]:
# STOCK
df_stock["c_anio"] = df_stock["c_anio"].astype(int)
df_stock["c_mes"] = df_stock["c_mes"].astype(int)
df_stock["c_articulo"] = df_stock["c_articulo"].astype(int)
df_stock["c_sucu_empr"] = df_stock["c_sucu_empr"].astype(int)
print("ANTES DE FILTRAR STOCK------------------------------>")
print(f"Parametros: {articulo} - {sucursal}")
print("Tipo de Datos: ", df_stock.dtypes)
dfs_filtrado = df_stock[(df_stock["c_articulo"] == articulo) & (df_stock["c_sucu_empr"] == sucursal)]

ANTES DE FILTRAR STOCK------------------------------>
Parametros: 5363 - 1
Tipo de Datos:  c_anio                    int64
c_mes                     int64
c_sucu_empr               int64
c_articulo                int64
q_dia1                  float64
q_dia2                  float64
q_dia3                  float64
q_dia4                  float64
q_dia5                  float64
q_dia6                  float64
q_dia7                  float64
q_dia8                  float64
q_dia9                  float64
q_dia10                 float64
q_dia11                 float64
q_dia12                 float64
q_dia13                 float64
q_dia14                 float64
q_dia15                 float64
q_dia16                 float64
q_dia17                 float64
q_dia18                 float64
q_dia19                 float64
q_dia20                 float64
q_dia21                 float64
q_dia22                 float64
q_dia23                 float64
q_dia24                 float64
q_dia25      

In [9]:
import traceback
import os
import time
from datetime import datetime

In [10]:
# Solo importa lo necesario desde el módulo de funciones
from funciones_forecast import (
    generar_grafico_json,


    get_execution_execute_by_status,
    get_full_parameters,
    update_execution,
    update_execution_execute,
    Open_Connection,
    Open_Diarco_Data,
    Close_Connection
)

# FUNCIONES LOCALES

In [11]:
dfv = df_ventas.copy(),
dfs =  df_stock.copy(),
articulo = 20
sucursal =  1
fecha_maxima = df_ventas['Fecha'].max()

In [12]:
dfv.info()

AttributeError: 'tuple' object has no attribute 'info'

In [13]:

df_filtrado = dfv[(dfv["Codigo_Articulo"] == articulo) & (dfv["Sucursal"] == sucursal)]
df_filtrado = df_filtrado[df_filtrado["Fecha"] >= (fecha_maxima - pd.Timedelta(days=50))]

# Validar unicidad por Fecha (clave para rolling)
if df_filtrado.duplicated(subset=["Fecha"]).any():
    raise ValueError(f"❌ Fechas duplicadas detectadas para Art {articulo} - Suc {sucursal}")

df_filtrado["Media_Movil"] = df_filtrado["Unidades"].rolling(window=7, min_periods=1).mean().fillna(0)
df_filtrado["Semana"] = df_filtrado["Fecha"].dt.to_period("W").astype(str)

df_semanal = df_filtrado.groupby("Semana")["Unidades"].sum().reset_index()
df_semanal["Semana_Num"] = df_filtrado.groupby("Semana")["Fecha"].min().reset_index()["Fecha"].dt.isocalendar().week.astype(int)
df_semanal["Media_Movil"] = df_semanal["Unidades"].rolling(window=7, min_periods=1).mean()

# Fechas de comparación
fecha_inicio_ultimos30 = fecha_maxima - pd.Timedelta(days=30)
fecha_inicio_previos30 = fecha_inicio_ultimos30 - pd.Timedelta(days=30)
fecha_inicio_anio_anterior = fecha_inicio_ultimos30 - pd.DateOffset(years=1)
fecha_fin_anio_anterior = fecha_inicio_previos30 - pd.DateOffset(years=1)

ventas_ultimos_30 = float(df_filtrado[df_filtrado["Fecha"] > fecha_inicio_ultimos30]["Unidades"].sum().item())
ventas_previos_30 = float(df_filtrado[
    (df_filtrado["Fecha"] > fecha_inicio_previos30) & (df_filtrado["Fecha"] <= fecha_inicio_ultimos30)
]["Unidades"].sum().item())

df_filtrado_anio_anterior = df_filtrado.copy()
df_filtrado_anio_anterior["Fecha"] = df_filtrado_anio_anterior["Fecha"] - pd.DateOffset(years=1)
ventas_mismo_periodo_anio_anterior = float(
    df_filtrado_anio_anterior[
        (df_filtrado_anio_anterior["Fecha"] > fecha_inicio_anio_anterior) &
        (df_filtrado_anio_anterior["Fecha"] <= fecha_fin_anio_anterior)
    ]["Unidades"].sum().item()
)

# Me traigo el stock filtrado
dfs_filtrado = dfs[(dfs["c_articulo"] == articulo) & (dfs["c_sucu_empr"] == sucursal)]

datos_stock = convertir_stock_diario_a_dict(dfs_filtrado) 

TypeError: tuple indices must be integers or slices, not str

In [ ]:
if os.path.exists(path_backup):
    df_backup = pd.read_csv(path_backup)
    procesados = set(zip(df_backup['Codigo_Articulo'], df_backup['Sucursal']))
    print(f"🔁 Recuperando avance previo: {len(procesados)} registros ya procesados")
else:
    df_backup = pd.DataFrame(columns=list(df_forecast.columns) + ['GRAFICO'])
    procesados = set()

nuevos = 0
total = len(df_forecast)

for i, row in df_forecast.iterrows():
    clave = (row['Codigo_Articulo'], row['Sucursal'])
    if clave in procesados:
        continue

    try:
        grafico = generar_grafico_json(
            df_ventas,
            df_stock,
            row['Codigo_Articulo'],
            row['Sucursal'],
            row['Forecast'],
            row['Average'],
            row['ventas_last'],
            row['ventas_previous'],
            row['ventas_same_year']
        )
        row_data = row.to_dict()
        row_data['GRAFICO'] = grafico
        df_backup = pd.concat([df_backup, pd.DataFrame([row_data])], ignore_index=True)
        nuevos += 1

        if nuevos % 50 == 0 or i == total - 1:
            df_backup.to_csv(path_backup, index=False)
            elapsed = round(time.time() - start_time, 2)
            print(f"🖼️ Procesados {nuevos} nuevos registros ({i+1}/{total}) - Tiempo: {elapsed} seg")
            with open(path_log, "a", encoding="utf-8") as log:
                log.write(f"[{datetime.now()}] {nuevos} registros procesados ({i+1}/{total}) - Tiempo: {elapsed} seg\n")

    except Exception as e:
        print(f"❌ Error procesando gráfico para Art {row['Codigo_Articulo']} - Suc {row['Sucursal']}: {e}")
        with open(path_log, "a", encoding="utf-8") as log:
            log.write(f"[{datetime.now()}] ERROR Art {row['Codigo_Articulo']} - Suc {row['Sucursal']}: {e}\n")
        continue

### LEIDAS DESDE EL REPOSITORIO DE FUNCIONES

In [ ]:

"""
Nombre del módulo: S30_GENERA_Grafico_Detalle.py

Descripción:
Partiendo de los datos extendidos con estado 30, se generan los gráficos de detalle para cada artículo y sucursal.
Se guarda el archivo CSV con los datos extendidos y los gráficos en formato base64.
Utiliza estad intermedio 35 miestras está graficando. Al finalizar se actualiza el estado a 40 en la base de datos.

Autor: EWE - Zeetrex
Fecha de creación: [2025-03-22]
"""
import traceback
import os
import time
from datetime import datetime

# Solo importar lo necesario desde el módulo de funciones
from funciones_forecast import (
    get_execution_execute_by_status,
    update_execution_execute,
    generar_grafico_base64,
    Open_Diarco_Data,
    Close_Connection,
    generar_grafico_json
)

import pandas as pd # uso localmente la lectura de archivos.
# import ace_tools_open as tools

from dotenv import dotenv_values
secrets = dotenv_values(".env")
folder = secrets["FOLDER_DATOS"]

# BLOQUE AGREGADO PARA INCORPORAR STOCK en formato DICCIONARIO
from datetime import date, datetime, timedelta

def convertir_stock_diario_a_dict(df_stock):
    """Convierte df_stock en un diccionario {fecha: cantidad}, solo hasta ayer y con fechas válidas."""
    def es_fecha_valida(anio, mes, dia):
        try:
            return date(anio, mes, dia)
        except ValueError:
            return None

    resultado = {}
    for _, row in df_stock.iterrows():
        anio = int(row['c_anio'])
        mes = int(row['c_mes'])

        for col in df_stock.columns:
            if col.startswith("q_dia"):
                dia = int(col.replace("q_dia", ""))
                fecha_valida = es_fecha_valida(anio, mes, dia)
                if fecha_valida and fecha_valida <= (datetime.now().date() - timedelta(days=1)):
                    resultado[fecha_valida.isoformat()] = row[col]
    return resultado

def generar_grafico_json(dfv, dfs, articulo, sucursal, Forecast, Average, ventas_last, ventas_previous, ventas_same_year):
    fecha_maxima = dfv["Fecha"].max()
    df_filtrado = dfv[(dfv["Codigo_Articulo"] == articulo) & (dfv["Sucursal"] == sucursal)]
    df_filtrado = df_filtrado[df_filtrado["Fecha"] >= (fecha_maxima - pd.Timedelta(days=50))]
    # Validar unicidad por Fecha (clave para rolling)
    if df_filtrado.duplicated(subset=["Fecha"]).any():
        raise ValueError(f"❌ Fechas duplicadas detectadas para Art {articulo} - Suc {sucursal}")

    df_filtrado["Media_Movil"] = df_filtrado["Unidades"].rolling(window=7, min_periods=1).mean().fillna(0)
    df_filtrado["Semana"] = df_filtrado["Fecha"].dt.to_period("W").astype(str)

    df_semanal = df_filtrado.groupby("Semana")["Unidades"].sum().reset_index()
    df_semanal["Semana_Num"] = df_filtrado.groupby("Semana")["Fecha"].min().reset_index()["Fecha"].dt.isocalendar().week.astype(int)
    df_semanal["Media_Movil"] = df_semanal["Unidades"].rolling(window=7, min_periods=1).mean()

    # Fechas de comparación
    fecha_inicio_ultimos30 = fecha_maxima - pd.Timedelta(days=30)
    fecha_inicio_previos30 = fecha_inicio_ultimos30 - pd.Timedelta(days=30)
    fecha_inicio_anio_anterior = fecha_inicio_ultimos30 - pd.DateOffset(years=1)
    fecha_fin_anio_anterior = fecha_inicio_previos30 - pd.DateOffset(years=1)

    ventas_ultimos_30 = float(df_filtrado[df_filtrado["Fecha"] > fecha_inicio_ultimos30]["Unidades"].sum().item())
    ventas_previos_30 = float(df_filtrado[
        (df_filtrado["Fecha"] > fecha_inicio_previos30) & (df_filtrado["Fecha"] <= fecha_inicio_ultimos30)
    ]["Unidades"].sum().item())

    df_filtrado_anio_anterior = df_filtrado.copy()
    df_filtrado_anio_anterior["Fecha"] = df_filtrado_anio_anterior["Fecha"] - pd.DateOffset(years=1)
    ventas_mismo_periodo_anio_anterior = float(
        df_filtrado_anio_anterior[
            (df_filtrado_anio_anterior["Fecha"] > fecha_inicio_anio_anterior) &
            (df_filtrado_anio_anterior["Fecha"] <= fecha_fin_anio_anterior)
        ]["Unidades"].sum().item()
    )
    
    # Me traigo el stock filtrado
    dfs_filtrado = dfs[(dfs["c_articulo"] == articulo) & (dfs["c_sucu_empr"] == sucursal)]
    
    datos_stock = convertir_stock_diario_a_dict(dfs_filtrado) 

    return {
        "articulo": int(articulo),
        "sucursal": int(sucursal),
        "fechas": df_filtrado["Fecha"].dt.strftime("%Y-%m-%d").tolist(),
        "unidades": [float(x) for x in df_filtrado["Unidades"]],
        "media_movil": [float(x) for x in df_filtrado["Media_Movil"]],
        "semana_num": df_semanal["Semana_Num"].astype(int).tolist(),
        "ventas_semanales": [float(x) for x in df_semanal["Unidades"]],
        "forecast": float(Forecast),
        "ventas_last": float(ventas_last),
        "ventas_previous": float(ventas_previous),
        "ventas_same_year": float(ventas_same_year),
        "average": float(Average),
        "ventas_ultimos_30": float(ventas_ultimos_30),
        "ventas_previos_30": float(ventas_previos_30),
        "ventas_anio_anterior": float(ventas_mismo_periodo_anio_anterior),
        "stock_diario": datos_stock   # ← aquí se integra el diccionario completo
    }



# RUTINA MEJORADA, Con RESGUARDO PARCIAL de Trabajo Realizado.
def insertar_graficos_forecast(algoritmo, name, id_proveedor):
    print("📊 Insertando Gráficos Forecast:   " + name)
    start_time = time.time()

    # Paths
    path_ventas = f'{folder}/{name}_Ventas.csv'
    path_forecast = f'{folder}/{algoritmo}_Pronostico_Extendido.csv'
    path_backup = f'{folder}/{algoritmo}_Pronostico_Extendido_Con_Graficos.csv'
    path_log = f'{folder}/log_graficos_{name}.txt'

    # Cargar historial de ventas
    df_ventas = pd.read_csv(path_ventas)
    df_ventas['Codigo_Articulo'] = df_ventas['Codigo_Articulo'].astype(int)
    df_ventas['Sucursal'] = df_ventas['Sucursal'].astype(int)
    df_ventas['Fecha'] = pd.to_datetime(df_ventas['Fecha'])

    # 🔄 Agrupar por Fecha, Código de Artículo y Sucursal, para consolidar múltiples precios
    df_ventas = (
        df_ventas
        .groupby(['Fecha', 'Codigo_Articulo', 'Sucursal'], as_index=False)
        .agg({'Unidades': 'sum'})
    )
    # Buscar filas duplicadas por clave compuesta
    duplicados = df_ventas[df_ventas.duplicated(subset=["Fecha", "Codigo_Articulo", "Sucursal"], keep=False)]
    # Ordenar para facilitar lectura
    duplicados = duplicados.sort_values(["Codigo_Articulo", "Sucursal", "Fecha"])
    # Mostrar o exportar
    print("⚠️ Filas duplicadas encontradas:")
    print(duplicados)

    # Cargar forecast extendido
    df_forecast = pd.read_csv(path_forecast)
    df_forecast.fillna(0, inplace=True)
    print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {name}")
    
    # Cargar STOCK por Proveedor
    """Consulta el stock y devuelve un dict {fecha: cantidad}, limitado a fechas válidas hasta ayer."""
    conn = Open_Diarco_Data()
    query_stock = f"""
        SELECT * FROM src.t710_estadis_stock s
        LEFT JOIN src.t050_articulos a
        ON  s.c_articulo = a.c_articulo
        WHERE c_anio * 100 + c_mes >= TO_CHAR(CURRENT_DATE - INTERVAL '1 month', 'YYYYMM')::INTEGER
        AND a.c_proveedor_primario = {id_proveedor};
    """
    df_stock = pd.read_sql(query_stock, conn)
    Close_Connection(conn)
    if df_stock.empty:
        print(f"⚠️ No se encontraron datos de stock para el proveedor {id_proveedor} en el mes actual.")
        return {}

    # Verificar si ya existe archivo con avances
    if os.path.exists(path_backup):
        df_backup = pd.read_csv(path_backup)
        procesados = set(zip(df_backup['Codigo_Articulo'], df_backup['Sucursal']))
        print(f"🔁 Recuperando avance previo: {len(procesados)} registros ya procesados")
    else:
        df_backup = pd.DataFrame(columns=list(df_forecast.columns) + ['GRAFICO'])
        procesados = set()

    nuevos = 0
    total = len(df_forecast)

    for i, row in df_forecast.iterrows():
        clave = (row['Codigo_Articulo'], row['Sucursal'])
        if clave in procesados:
            continue

        try:
            grafico = generar_grafico_json(
                df_ventas,
                df_stock,
                row['Codigo_Articulo'],
                row['Sucursal'],
                row['Forecast'],
                row['Average'],
                row['ventas_last'],
                row['ventas_previous'],
                row['ventas_same_year']
            )
            row_data = row.to_dict()
            row_data['GRAFICO'] = grafico
            df_backup = pd.concat([df_backup, pd.DataFrame([row_data])], ignore_index=True)
            nuevos += 1

            if nuevos % 50 == 0 or i == total - 1:
                df_backup.to_csv(path_backup, index=False)
                elapsed = round(time.time() - start_time, 2)
                print(f"🖼️ Procesados {nuevos} nuevos registros ({i+1}/{total}) - Tiempo: {elapsed} seg")
                with open(path_log, "a", encoding="utf-8") as log:
                    log.write(f"[{datetime.now()}] {nuevos} registros procesados ({i+1}/{total}) - Tiempo: {elapsed} seg\n")

        except Exception as e:
            print(f"❌ Error procesando gráfico para Art {row['Codigo_Articulo']} - Suc {row['Sucursal']}: {e}")
            with open(path_log, "a", encoding="utf-8") as log:
                log.write(f"[{datetime.now()}] ERROR Art {row['Codigo_Articulo']} - Suc {row['Sucursal']}: {e}\n")
            continue

    # Guardar completo al final
    df_backup.to_csv(path_backup, index=False)
    elapsed = round(time.time() - start_time, 2)
    print(f"✅ Finalizado: {name} - Total nuevos: {nuevos} - Tiempo total: {elapsed} segundos")
    with open(path_log, "a", encoding="utf-8") as log:
        log.write(f"[{datetime.now()}] FINALIZADO: {nuevos} registros nuevos - Tiempo total: {elapsed} seg\n")

    return df_backup

# Punto de entrada
if __name__ == "__main__":
    fes = get_execution_execute_by_status(30)

    # Filtrar registros con supply_forecast_execution_status_id = 30  #FORECAST con DFATOSK
    for index, row in fes[fes["fee_status_id"].isin([30])].iterrows():
        algoritmo = row["name"] 
        name = algoritmo.split('_ALGO')[0]
        execution_id = row["forecast_execution_id"]
        id_proveedor = row["ext_supplier_code"]
        forecast_execution_execute_id = row["forecast_execution_execute_id"]

        print(f"Algoritmo: {algoritmo}  - Name: {name}  exce_id: {execution_id}  Proveedor: {id_proveedor}")

        try:
            # Estado intermedio: 35 (procesando gráficos)
            print(f"🛠 Marcando como 'Procesando Gráficos' para {execution_id}")
            update_execution_execute(forecast_execution_execute_id, supply_forecast_execution_status_id=35)
            print(f"🛠 Iniciando graficación para {execution_id}...")

            # Generación del dataframe extendido con gráficos
            df_merged = insertar_graficos_forecast(algoritmo, name, id_proveedor)

            # Guardar el CSV con datos extendidos y gráficos
            file_path = f"{folder}/{algoritmo}_Pronostico_Extendido_FINAL.csv"
            df_merged.to_csv(file_path, index=False)
            print(f"📁 Archivo guardado correctamente: {file_path}")

            # ✅ Solo si todo fue exitoso, actualizamos el estado a 40
            update_execution_execute(forecast_execution_execute_id, supply_forecast_execution_status_id=40)
            print(f"✅ Estado actualizado a 40 para {execution_id}")

        except Exception as e:
            traceback.print_exc()
            print(f"❌ Error procesando {name}: {e}")
            
            log_path = os.path.join(folder, "errores_s30.log")
            with open(log_path, "a", encoding="utf-8") as log_file:
                log_file.write(f"[{name}] ID: {execution_id} - ERROR: {str(e)}\n")
            
            continue




Algoritmo: 20_MOLINOS_RIO_ALGO_01  - Name: 20_MOLINOS_RIO  exce_id: 5fb98714-9424-4ef5-8c10-f46446ce30b0  Proveedor: 20
🛠 Marcando como 'Procesando Gráficos' para 5fb98714-9424-4ef5-8c10-f46446ce30b0
🛠 Iniciando graficación para 5fb98714-9424-4ef5-8c10-f46446ce30b0...
📊 Insertando Gráficos Forecast:   20_MOLINOS_RIO
-> Datos Recuperados del CACHE: 20, Label: 20_MOLINOS_RIO
❌ Error procesando gráfico para Art 3796 - Suc 1: cannot reindex on an axis with duplicate labels
❌ Error procesando gráfico para Art 3796 - Suc 2: cannot reindex on an axis with duplicate labels
❌ Error procesando gráfico para Art 3796 - Suc 3: cannot reindex on an axis with duplicate labels
❌ Error procesando gráfico para Art 3796 - Suc 5: cannot reindex on an axis with duplicate labels
❌ Error procesando gráfico para Art 3796 - Suc 7: cannot reindex on an axis with duplicate labels
❌ Error procesando gráfico para Art 3796 - Suc 9: cannot reindex on an axis with duplicate labels
❌ Error procesando gráfico para Art 

KeyboardInterrupt: 